In [0]:
from pyspark.sql import functions as F
import re

bronze_table = "hospital_poc.bronze.patient_notes"
silver_table = "hospital_poc.silver.patient_transcripts"

df_bronze = spark.table(bronze_table)

print("Bronze record count:", df_bronze.count())

Bronze record count: 1000


In [0]:
df_bronze.printSchema()

root
 |-- id: long (nullable = true)
 |-- length: string (nullable = true)
 |-- note: string (nullable = true)
 |-- specialty: string (nullable = true)
 |-- version: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
#define section headings
sections = [
    "Patient Name",
    "Age/Sex",
    "UHID",
    "Admission No",
    "Address",
    "Admission Date",
    "Discharge Date",
    "Treating Consultant",
    "Chief Complaints",
    "History of Present Illness",
    "HPI",
    "Past History",
    "Physical Examination",
    "Investigations",
    "Diagnosis",
    "Course in Hospital",
    "Socio-Economic Status",
    "Traditional Medicine History",
    "Condition at Discharge",
    "Advice on Discharge"
]

In [0]:
def extract_section(note, section_name, next_sections):

    if not note:
        return ""

    next_pattern = "|".join(
        re.escape(section)
        for section in next_sections
    )

    pattern = (
        rf"{re.escape(section_name)}\s*:?\s*"
        rf"(.*?)"
        rf"(?=\n\s*(?:{next_pattern})\s*:|\Z)"
    )

    match = re.search(
        pattern,
        note,
        flags=re.IGNORECASE | re.DOTALL
    )

    if match:
        return match.group(1).strip()

    return ""

In [0]:
def clean_section(text):

    if not text:
        return ""

    text = text.strip()

    # Remove signature section if it appears after discharge advice
    text = re.split(
        r"\n\s*Dr\.\s+.*?\n\s*Signature",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )[0]

    # Remove trailing Signature
    text = re.sub(
        r"\n\s*Signature\s*$",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text.strip()

In [0]:
test_note = (
    df_bronze
    .select("note")
    .first()["note"]
)

for section in sections:

    value = extract_section(
        test_note,
        section,
        sections
    )

    value = clean_section(value)

    if value:
        print(f"\n--- {section} ---")
        print(value)


--- Patient Name ---
Baby Saanvi

--- Age/Sex ---
68/F

--- UHID ---
632376936

--- Admission No ---
ADM-20260462

--- Address ---
Patna, Bihar

--- Admission Date ---
07/03/2026 08:00 AM

--- Discharge Date ---
14/03/2026 01:00 PM

--- Treating Consultant ---
Dr. Naresh Trehan (Cardiac Surgeon), Oncology

--- Chief Complaints ---
Unexplained weight loss and fatigue for 8 months.

--- History of Present Illness ---
Patient noticed a gradual loss of appetite.

--- Past History ---
No history of major illness.

--- Physical Examination ---
2x2 cm hard lump in the upper outer quadrant of the left breast.
- PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -

--- Investigations ---
CBC shows anemia. PET scan shows metastasis.

--- Diagnosis ---
Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).

--- Course in Hospital ---
Underwent mastectomy followed by chemotherapy.

--- Socio-Economic Status ---
Class IV as per modified Kuppusw

In [0]:
def clean_chief_complaints(text):

    if not text:
        return ""

    # Remove embedded HPI portion
    text = re.split(
        r"\n\s*History of Present Illness\s*:",
        text,
        flags=re.IGNORECASE
    )[0]

    return text.strip()

In [0]:
chief_complaints = extract_section(
    test_note,
    "Chief Complaints",
    sections
)

hpi = extract_section(
    test_note,
    "History of Present Illness",
    sections
)

chief_complaints = clean_chief_complaints(chief_complaints)
hpi = clean_section(hpi)

print("Chief Complaints:")
print(chief_complaints)

print("\nHPI:")
print(hpi)

Chief Complaints:
Unexplained weight loss and fatigue for 8 months.

HPI:
Patient noticed a gradual loss of appetite.


In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

In [0]:
def get_section(note, section_name):

    value = extract_section(
        note,
        section_name,
        sections
    )

    return clean_section(value)


get_section_udf = udf(
    get_section,
    StringType()
)

In [0]:
df_silver = df_bronze

for section in sections:

    column_name = (
        section.lower()
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
    )

    df_silver = df_silver.withColumn(
        column_name,
        get_section_udf(
            F.col("note"),
            F.lit(section)
        )
    )

In [0]:
#fix the Chief Complaints column
df_silver = df_silver.withColumn(
    "chief_complaints",
    F.regexp_extract(
        F.col("chief_complaints"),
        r"(?is)(.*?)(?=\n\s*History of Present Illness\s*:|\Z)",
        1
    )
)

In [0]:
df_silver = df_silver.withColumn(
    "chief_complaints",
    F.trim(F.col("chief_complaints"))
)

In [0]:
df_silver = df_silver.withColumn(
    "hpi_final",
    F.when(
        F.trim(F.col("history_of_present_illness")) != "",
        F.col("history_of_present_illness")
    ).otherwise(
        F.col("hpi")
    )
)

In [0]:
#narrative transcript
df_silver = df_silver.withColumn(
    "transcript",
    F.concat_ws(
        " ",
        
        F.concat(
            F.lit("Patient ID: "),
            F.col("id").cast("string"),
            F.lit(".")
        ),

        F.concat(
            F.lit("Specialty: "),
            F.col("specialty"),
            F.lit(".")
        ),

        F.when(
            F.trim(F.col("patient_name")) != "",
            F.concat(
                F.lit("Patient name: "),
                F.col("patient_name"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("age_sex")) != "",
            F.concat(
                F.lit("Age/Sex: "),
                F.col("age_sex"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("chief_complaints")) != "",
            F.concat(
                F.lit("The patient presented with "),
                F.col("chief_complaints"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("hpi_final")) != "",
            F.concat(
                F.lit("The history of present illness was "),
                F.col("hpi_final"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("past_history")) != "",
            F.concat(
                F.lit("Relevant past history included "),
                F.col("past_history"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("physical_examination")) != "",
            F.concat(
                F.lit("Physical examination showed "),
                F.col("physical_examination"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("investigations")) != "",
            F.concat(
                F.lit("Investigations revealed "),
                F.col("investigations"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("diagnosis")) != "",
            F.concat(
                F.lit("The diagnosis was "),
                F.col("diagnosis"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("course_in_hospital")) != "",
            F.concat(
                F.lit("During the hospital stay, "),
                F.col("course_in_hospital"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("condition_at_discharge")) != "",
            F.concat(
                F.lit("At discharge, the patient's condition was "),
                F.col("condition_at_discharge"),
                F.lit(".")
            )
        ),

        F.when(
            F.trim(F.col("advice_on_discharge")) != "",
            F.concat(
                F.lit("The patient was advised "),
                F.col("advice_on_discharge"),
                F.lit(".")
            )
        )
    )
)

In [0]:
df_silver_final = df_silver.select(
    
    F.col("id").alias("patient_id"),
    
    "specialty",
    
    "patient_name",
    "age_sex",
    "uhid",
    "admission_no",
    "admission_date",
    "discharge_date",
    "treating_consultant",
    
    "chief_complaints",
    F.col("hpi_final").alias("hpi"),
    
    "past_history",
    "physical_examination",
    "investigations",
    "diagnosis",
    "course_in_hospital",
    
    "socio_economic_status",
    "traditional_medicine_history",
    
    "condition_at_discharge",
    "advice_on_discharge",
    
    "transcript",
    
    F.current_timestamp().alias("created_at")
)

In [0]:
display(
    df_silver_final.select(
        "patient_id",
        "specialty",
        "chief_complaints",
        "hpi",
        "diagnosis",
        "transcript"
    ).limit(5)
)

patient_id,specialty,chief_complaints,hpi,diagnosis,transcript
1,oncology,Unexplained weight loss and fatigue for 8 months.,Patient noticed a gradual loss of appetite.,Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).,"Patient ID: 1. Specialty: oncology. Patient name: Baby Saanvi. Age/Sex: 68/F. The patient presented with Unexplained weight loss and fatigue for 8 months.. The history of present illness was Patient noticed a gradual loss of appetite.. Relevant past history included No history of major illness.. Physical examination showed 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CBC shows anemia. PET scan shows metastasis.. The diagnosis was Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).. During the hospital stay, Underwent mastectomy followed by chemotherapy.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Regular follow-up in the oncology OPD. - Dawa time pe lena.."
2,cardiology,Palpitations and dizziness for 3 days.,"Substernal pain on exertion, with diaphoresis. Patient ko fever tha.",Atrial Fibrillation (ICD: I48.91).,"Patient ID: 2. Specialty: cardiology. Patient name: Debashish Roy. Age/Sex: 85/F. The patient presented with Palpitations and dizziness for 3 days.. The history of present illness was Substernal pain on exertion, with diaphoresis. Patient ko fever tha.. Relevant past history included Coronary artery disease, post-stent.. Physical examination showed BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD.. The diagnosis was Atrial Fibrillation (ICD: I48.91).. During the hospital stay, Antiarrhythmics given, rhythm stabilized.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna.."
3,general_medicine,Abdominal pain for 22 days.,"Productive cough, no hemoptysis.",Community-Acquired Pneumonia (ICD: J18.9).,"Patient ID: 3. Specialty: general_medicine. Patient name: Jayesh Patel. Age/Sex: 85/F. The patient presented with Abdominal pain for 22 days.. The history of present illness was Productive cough, no hemoptysis.. Relevant past history included Diabetes on insulin.. Physical examination showed Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed Endoscopy: Gastritis.. The diagnosis was Community-Acquired Pneumonia (ICD: J18.9).. During the hospital stay, IV antibiotics. Fever subsided.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Diet: Bland; Follow-up: 1 week.."
4,pediatrics,Loose motions aur vomiting for 17 days.,"High fever, measles-like rash.",Viral Exanthem (ICD: B09).,"Patient ID: 4. Specialty: pediatrics. Patient name: Imran Khan. Age/Sex: 35/M. The patient presented with Loose motions aur vomiting for 17 days.. The history of present illness was High fever, measles-like rash.. Relevant past history included Recent URI.. Physical examination showed Dehydrated, abdomen soft. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CRP: Positive.. The diagnosis was Viral Exanthem (ICD: B09).. During the hospital stay, ORS and IV fluids. Zinc given.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Diet: Breastfeed; Follow-up: 5 days. - report leke aana.."
5,pediatrics,Fever and rash for 27 days.,"Acute, with mild dehydration.",Viral Exanthem (ICD: B09).,"Patient ID: 5. Specialty: pediatrics. Patient name: Sanjay Yada

In [0]:
silver_table = "hospital_poc.silver.patient_transcripts"

(
    df_silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
df_check = spark.table(silver_table)

print(
    "Silver record count:",
    df_check.count()
)

display(
    df_check.limit(5)
)

Silver record count: 1000


patient_id,specialty,patient_name,age_sex,uhid,admission_no,admission_date,discharge_date,treating_consultant,chief_complaints,hpi,past_history,physical_examination,investigations,diagnosis,course_in_hospital,socio_economic_status,traditional_medicine_history,condition_at_discharge,advice_on_discharge,transcript,created_at
1,oncology,Baby Saanvi,68/F,632376936,ADM-20260462,07/03/2026 08:00 AM,14/03/2026 01:00 PM,"Dr. Naresh Trehan (Cardiac Surgeon), Oncology",Unexplained weight loss and fatigue for 8 months.,Patient noticed a gradual loss of appetite.,No history of major illness.,"2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -",CBC shows anemia. PET scan shows metastasis.,Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).,Underwent mastectomy followed by chemotherapy.,Class IV as per modified Kuppuswamy scale.,Used Tulsi for cough.,"Stable, afebrile.",- Regular follow-up in the oncology OPD. - Dawa time pe lena.,"Patient ID: 1. Specialty: oncology. Patient name: Baby Saanvi. Age/Sex: 68/F. The patient presented with Unexplained weight loss and fatigue for 8 months.. The history of present illness was Patient noticed a gradual loss of appetite.. Relevant past history included No history of major illness.. Physical examination showed 2x2 cm hard lump in the upper outer quadrant of the left breast. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed CBC shows anemia. PET scan shows metastasis.. The diagnosis was Metastatic Adenocarcinoma of unknown primary (ICD: C80.0).. During the hospital stay, Underwent mastectomy followed by chemotherapy.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Regular follow-up in the oncology OPD. - Dawa time pe lena..",2026-09-23T09:36:45.695Z
2,cardiology,Debashish Roy,85/F,247712229,ADM-20268090,09/03/2026 09:00 AM,14/03/2026 02:00 PM,"Dr. Anita Sharma, Cardiology",Palpitations and dizziness for 3 days.,"Substernal pain on exertion, with diaphoresis. Patient ko fever tha.","Coronary artery disease, post-stent.","BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -",ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD.,Atrial Fibrillation (ICD: I48.91).,"Antiarrhythmics given, rhythm stabilized.",Class III as per modified Kuppuswamy scale.,Used Tulsi for cough.,"Stable, afebrile.","- Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna.","Patient ID: 2. Specialty: cardiology. Patient name: Debashish Roy. Age/Sex: 85/F. The patient presented with Palpitations and dizziness for 3 days.. The history of present illness was Substernal pain on exertion, with diaphoresis. Patient ko fever tha.. Relevant past history included Coronary artery disease, post-stent.. Physical examination showed BP: 160/102 mmHg, Heart: S1/S2 normal, no murmur. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenopathy: -, Edema: -. Investigations revealed ECG: ST elevation; Trop-I: Positive; Echo: EF 45%; Angiogram: 52% blockage in LAD.. The diagnosis was Atrial Fibrillation (ICD: I48.91).. During the hospital stay, Antiarrhythmics given, rhythm stabilized.. At discharge, the patient's condition was Stable, afebrile.. The patient was advised - Meds: Tab. Aspirin 75mg OD, Tab. Atorvastatin 40mg HS. - pura course complete karna..",2026-09-23T09:36:45.695Z
3,general_medicine,Jayesh Patel,85/F,805408206,ADM-20267292,03/03/2026 09:00 AM,10/03/2026 03:00 PM,"Dr. Rajesh Kumar, General_medicine",Abdominal pain for 22 days.,"Productive cough, no hemoptysis.",Diabetes on insulin.,"Temp: 102°F; Chest: Crepitations. - PICCKLE: Pallor: +, Icterus: -, Cyanosis: -, Clubbing: -, Koilonychia: -, Lymphadenop

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM hospital_poc.silver.patient_transcripts;

total_records
1000


In [0]:
%sql
SELECT COUNT(DISTINCT patient_id) AS unique_patients
FROM hospital_poc.silver.patient_transcripts;

unique_patients
1000


In [0]:
%sql
SELECT COUNT(*) AS empty_transcripts
FROM hospital_poc.silver.patient_transcripts
WHERE transcript IS NULL
   OR TRIM(transcript) = '';

empty_transcripts
0


In [0]:
%sql
--transcript length
SELECT
    MIN(LENGTH(transcript)) AS min_length,
    MAX(LENGTH(transcript)) AS max_length,
    ROUND(AVG(LENGTH(transcript)), 2) AS avg_length
FROM hospital_poc.silver.patient_transcripts;

min_length,max_length,avg_length
666,983,791.12
